In [ ]:
%cd ../../..

In [ ]:
import pandas as pd
import numpy as np
import holidays
from darts import TimeSeries
from darts.models import Prophet, RegressionModel
from darts.metrics import mae, mse, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
paths = [
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Viikuna 2023.csv",
    "data/raw/pos/Sold lunches Viikuna 2024.csv",
    "data/raw/pos/Sold lunches.csv"
]


list_df = []
for path in paths:
    df = pd.read_csv(path, delimiter=';')
    df.columns = ['Date', 'Receipt time', 'Restaurant', 'Food Category', 'Dish', 'pcs', 'Hiilijalanjälki']

    list_df.append(df)

data = pd.concat(list_df)

data.head()

In [4]:
data['Restaurant'] = data['Restaurant'].map({
    '610 Physicum': 'Physicum',
    '600 Chemicum': 'Chemicum',
    '620 Exactum': 'Exactum',
    '570 Viikuna': 'Viikuna'
})

data['pcs'] = data['pcs'].replace(' ', '0').astype(float)
data['Date'] = pd.to_datetime(data['Date'], format="%d.%m.%Y")


In [5]:
date_range = pd.date_range(start=data['Date'].min(), end=data['Date'].max())

restaurants = data['Restaurant'].unique()

In [ ]:
predictions = {}

for restaurant in restaurants:
    # Filter data for the current restaurant
    restaurant_sales = data[data['Restaurant'] == restaurant]
    
    # Ensure there are no duplicate dates by aggregating on 'Date'
    restaurant_sales = restaurant_sales.groupby('Date')['pcs'].sum().reset_index()
    
    # Reindex to fill missing dates for this restaurant
    restaurant_sales = restaurant_sales.set_index('Date').reindex(date_range, fill_value=0).reset_index()
    
    # Add back the 'Restaurant' column
    restaurant_sales['Restaurant'] = restaurant

    restaurant_sales.columns = ['Date', 'pcs', 'Restaurant']
    
    # Add Finland holidays
    fin_holidays = holidays.Finland(years=restaurant_sales['Date'].dt.year.unique())
    restaurant_sales['Holiday'] = restaurant_sales['Date'].isin(fin_holidays).astype(int)
    
    # Add exam weeks
    exam_weeks = [
        ('2023-03-06', '2023-03-12'),  
        ('2023-05-01', '2023-05-07'),
        ('2023-10-23', '2023-10-29'),
        ('2023-12-18', '2023-12-24'),
        ('2024-03-04', '2024-03-10'),
        ('2024-05-06', '2024-05-12')
    ]
    
    restaurant_sales['ExamWeek'] = 0
    for start, end in exam_weeks:
        restaurant_sales.loc[(restaurant_sales['Date'] >= start) & (restaurant_sales['Date'] <= end), 'ExamWeek'] = 1
    
    # Additional features
    restaurant_sales['Lag_1'] = restaurant_sales['pcs'].shift(-1)
    restaurant_sales['Lag_7'] = restaurant_sales['pcs'].shift(-7)
    restaurant_sales.fillna(0, inplace=True)
    restaurant_sales['DayOfWeek'] = restaurant_sales['Date'].dt.dayofweek
    restaurant_sales['Month'] = restaurant_sales['Date'].dt.month
    restaurant_sales['Year'] = restaurant_sales['Date'].dt.year
    
    # Create TimeSeries objects for target (pcs) and covariates
    ts_pcs = TimeSeries.from_dataframe(restaurant_sales, 'Date', 'pcs')
    ts_covariates = TimeSeries.from_dataframe(restaurant_sales, 'Date', 
                                              ['Holiday', 'ExamWeek', 'DayOfWeek', 'Month', 'Year', 
                                               'Lag_1', 'Lag_7'])

    model = RegressionModel(
        lags=7,  # Using the last 7 days of sales to predict the next day
        lags_future_covariates=[0],  # Using today's holiday and exam week info
        model=RandomForestRegressor()
    )

    model.fit(ts_pcs, future_covariates=ts_covariates)
    
    # Create future date range till January 2025
    future_dates = pd.date_range(start=restaurant_sales['Date'].max() + pd.Timedelta(days=1), end='2025-07-01')
    
    # Add Finland holidays for the forecast horizon
    future_holidays = holidays.Finland(years=future_dates.year.unique())
    future_holiday_series = pd.Series(future_dates.isin(future_holidays), index=future_dates)
    
    # Add exam weeks for the forecast horizon
    future_exam_weeks = [
        ('2024-03-04', '2024-03-10'),
        ('2024-05-06', '2024-05-12')
    ]
    future_exam_series = pd.Series(0, index=future_dates)
    for start, end in future_exam_weeks:
        future_exam_series.loc[(future_dates >= start) & (future_dates <= end)] = 1
    
    # Future covariates dataframe
    future_covariates_df = pd.DataFrame({
        'Date': future_dates,
        'Holiday': future_holiday_series.astype(int),
        'ExamWeek': future_exam_series.astype(int),
        'DayOfWeek': future_dates.dayofweek,
        'Month': future_dates.month,
        'Year': future_dates.year
    })

    # Create TimeSeries objects for the entire historical data and future covariates
    ts_pcs_full = TimeSeries.from_dataframe(restaurant_sales, 'Date', 'pcs')
    ts_covariates_full = TimeSeries.from_dataframe(restaurant_sales, 'Date', ['Holiday', 'ExamWeek', 'DayOfWeek', 'Month', 'Year'])
    
    # Future TimeSeries for prediction
    future_covariates_ts = TimeSeries.from_dataframe(future_covariates_df, 'Date', ['Holiday', 'ExamWeek', 'DayOfWeek', 'Month', 'Year'])

    model.fit(ts_pcs_full, future_covariates=ts_covariates_full)

    future_prediction = model.predict(len(future_covariates_ts), future_covariates=future_covariates_ts)

    predictions[restaurant] = future_prediction

    plt.figure(figsize=(10, 6))
    ts_pcs.plot(label='Actual Sales')
    future_prediction.plot(label='Predicted Sales (Future)')
    plt.title(f'Sales Forecast for Restaurant {restaurant} until January 2025')
    plt.legend()
    plt.show()

In [7]:
date_start = '2024-09-01'
date_end = '2025-07-01'

dates = pd.bdate_range(date_start, date_end)

In [11]:
concated = []

for restaurant, prediction in predictions.items():
    pred = prediction.pd_dataframe().reset_index().rename(columns={'Date': 'date'})
    pred = pred[pred['date'].isin(dates)]

    pred['restaurant'] = restaurant

    concated.append(pred)

out = pd.concat(concated)
out['restaurant'] = out['restaurant'].map({'Chemicum': 'che', 'Exactum': 'exa', 'Physicum': 'phy', 'Viikuna': 'vik'})
out.to_excel("data/processed/phase_4/dim_pieces_whole.xlsx", index=False)

In [ ]:
out[(out['date'] >= '2025-05-01') & (out['restaurant'] == 'vik')]

uti